In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch02-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "02-04-standardization-and-salts.py"

# 分開看去鹽與電荷處理

先看兩個概念例，再檢查 ESOL 的真實資料；候選結構不覆寫原始量測。練習可加在分析格末尾，再執行最後一格。

In [ ]:
%%writefile "{script}"
from pathlib import Path
import os
import deepchem as dc
import pandas as pd
from rdkit import Chem, rdBase
from rdkit.Chem import Draw
from rdkit.Chem.MolStandardize import rdMolStandardize as std

# 概念示意 — 實際應用請使用下方 DeepChem 版本
examples = [("Sodium benzoate", "O=C([O-])c1ccccc1.[Na+]"),
            ("Tetramethylammonium chloride", "C[N+](C)(C)C.[Cl-]")]
uncharger = std.Uncharger()
def stages(mol):
    cleaned = std.Cleanup(mol)
    parent = std.FragmentParent(cleaned, skipStandardize=True)
    candidate = uncharger.uncharge(parent)
    return cleaned, parent, candidate

def key(mol):
    return Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)

panels, legends = [], []
for name, smiles in examples:
    mol = Chem.MolFromSmiles(smiles)
    cleaned, parent, candidate = stages(mol)
    for stage, structure in zip(["Cleanup", "FragmentParent", "Uncharger"], [cleaned, parent, candidate]):
        panels.append(structure)
        legends.append(f"From {name}\n{stage}: charge {Chem.GetFormalCharge(structure):+d}")
        print(name, stage, key(structure))
    assert key(mol) == key(cleaned)
assert Chem.GetFormalCharge(panels[2]) == 0
assert Chem.GetFormalCharge(panels[5]) == 1

# ESOL：1128 筆，目標為實測 log10(S / (mol/L))；DOI: 10.1021/ci034243x。
# 僅檢查結構，不訓練模型，因此不切分；transformers=[] 保留原始標籤尺度。
_, (data,), transformers = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False, data_dir=os.environ.get("DEEPCHEM_DATA_DIR"))
assert not transformers
rows = []
for i, (smiles, value, weight) in enumerate(zip(data.ids, data.y[:, 0], data.w[:, 0]), 1):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"ESOL row {i}: cannot parse")
    cleaned, parent, candidate = stages(mol)
    rows.append({"record_id": f"ESOL-row-{i}", "original_smiles": smiles,
                 "logS": float(value), "weight": float(weight), "original_key": key(mol),
                 "cleanup_key": key(cleaned), "parent_candidate": key(parent),
                 "uncharge_candidate": key(candidate),
                 "fragments": len(Chem.GetMolFrags(mol))})
audit = pd.DataFrame(rows)
# 候選欄只供審查；保留完整原始結構、列號及量測，不覆寫也不合併資料列。
audit["cleanup_changed"] = audit.original_key != audit.cleanup_key
audit["parent_changed"] = audit.cleanup_key != audit.parent_candidate
audit["uncharge_changed"] = audit.parent_candidate != audit.uncharge_candidate
print("RDKit", rdBase.rdkitVersion)
print("rows", len(audit), "multifragment", int((audit.fragments > 1).sum()))
print(audit[["cleanup_changed", "parent_changed", "uncharge_changed"]].sum().to_string())
changed = audit.loc[audit[["cleanup_changed", "parent_changed", "uncharge_changed"]].any(axis=1)]
print(changed[["record_id", "logS", "original_key", "cleanup_key", "parent_candidate", "uncharge_candidate"]].to_string(index=False))
assert len(audit) == len(data.ids)
assert audit.original_smiles.tolist() == data.ids.tolist()
assert audit.logS.tolist() == data.y[:, 0].tolist()
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
figure = Draw.MolsToGridImage(panels, molsPerRow=3, subImgSize=(340, 240), legends=legends)
figure.save(out / "02-04-standardization-and-salts.png")


In [ ]:
from IPython.display import Image, display

run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.removeprefix("This module requires PyTorch to be installed.\n").strip())
figure_dir = Path("../../figures") if Path("../../outline.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "02-04-standardization-and-salts.png")))